In [28]:
!pip install -q pandas numpy requests beautifulsoup4 tqdm lxml

In [29]:
import pandas as pd
import numpy as np
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
from tqdm import tqdm

In [30]:
OUTPUT_FILE = "cafe_drinks_scrape.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

KATALOG_URLS = [
    {
        "url": "https://menukuliner.net/katalog/coffee-boba",
        "raw_category": "coffee / boba",
        "city": "unknown",
        "keyword": "coffee boba"
    },
    {
        "url": "https://menukuliner.net/katalog/kopi-boba",
        "raw_category": "coffee / boba",
        "city": "unknown",
        "keyword": "kopi boba"
    },
    {
        "url": "https://menukuliner.net/katalog/boba-kopi",
        "raw_category": "boba / coffee",
        "city": "unknown",
        "keyword": "boba kopi"
    },
    {
        "url": "https://menukuliner.net/katalog/kopi",
        "raw_category": "coffee",
        "city": "unknown",
        "keyword": "kopi"
    },
    {
        "url": "https://menukuliner.net/katalog/boba",
        "raw_category": "boba",
        "city": "unknown",
        "keyword": "boba"
    },
    {
        "url": "https://menukuliner.net/katalog/matcha",
        "raw_category": "matcha",
        "city": "unknown",
        "keyword": "matcha"
    },
    {
        "url": "https://menukuliner.net/katalog/milk-tea",
        "raw_category": "milk tea",
        "city": "unknown",
        "keyword": "milk tea"
    },
    {
        "url": "https://menukuliner.net/katalog/thai-tea",
        "raw_category": "thai tea",
        "city": "unknown",
        "keyword": "thai tea"
    },
    {
        "url": "https://menukuliner.net/katalog/es-kopi-susu",
        "raw_category": "coffee",
        "city": "unknown",
        "keyword": "es kopi susu"
    },
]

MAX_LINKS = 75
REQUEST_DELAY = 1

In [31]:
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_price(text):
    if text is None or pd.isna(text):
        return np.nan

    text = str(text)
    text = re.sub(r"[^0-9]", "", text)

    if text == "":
        return np.nan

    return int(text)


def is_price(text):
    text = clean_text(text)
    return bool(re.search(r"Rp\s*[0-9][0-9\.\,]*", text))


def is_valid_menu_name(text):
    text = clean_text(text)
    lower = text.lower()

    if len(text) < 3:
        return False

    if len(text) > 120:
        return False

    if lower.startswith("rp"):
        return False

    if re.fullmatch(r"[0-9\.\,\s]+", text):
        return False

    noise_words = [
        "nama menu harga",
        "nama menu",
        "harga",
        "harga menu",
        "daftar harga",
        "delivery",
        "gofood",
        "gojek",
        "grabfood",
        "shopeefood",
        "menukuliner",
        "restaurant",
        "restoran",
        "promo",
        "diskon",
        "review",
        "rating",
        "alamat",
        "jam buka",
        "telepon",
        "cukup merogoh",
        "dibanderol",
        "disajikan",
        "berkisar",
        "pilihan menu",
        "siapkan uang",
        "tidak mahal",
        "untuk menyantap",
        "harga yang dibanderol",
        "anda bisa",
        "di bawah ini",
        "berikut ini",
        "terbaru",
        "halaman",
        "lihat",
        "baca juga",
    ]

    if any(word in lower for word in noise_words):
        return False

    return True


def infer_raw_category(text):
    text = str(text).lower()

    if any(k in text for k in ["boba", "brown sugar", "bubble"]):
        return "boba"

    if any(k in text for k in ["milk tea", "thai tea", "cheese tea"]):
        return "milk tea"

    if any(k in text for k in ["matcha", "green tea", "hojicha"]):
        return "matcha / tea"

    if any(k in text for k in ["kopi", "coffee", "latte", "americano", "espresso", "cappuccino", "macchiato", "mocha"]):
        return "coffee"

    if any(k in text for k in ["choco", "coklat", "milo", "red velvet", "taro"]):
        return "non-coffee drink"

    return "cafe drinks"


def infer_city_from_text(text):
    text = str(text).lower()

    cities = [
        "jakarta", "semarang", "bandung", "yogyakarta", "surabaya",
        "medan", "tangerang", "bekasi", "bogor", "depok",
        "malang", "solo", "denpasar", "balikpapan", "makassar"
    ]

    for city in cities:
        if city in text:
            return city.title()

    return "Unknown"

In [32]:
def collect_menu_links():
    collected = []

    for katalog in tqdm(KATALOG_URLS, desc="Collecting menu links"):
        url = katalog["url"]

        try:
            response = requests.get(url, headers=HEADERS, timeout=30)

            if response.status_code != 200:
                print(f"Skip {url} | status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "lxml")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if "/menu/" not in href:
                    continue

                full_url = urljoin("https://menukuliner.net", href)

                if "menukuliner.net/menu/" not in full_url:
                    continue

                collected.append({
                    "source_platform": "MenuKuliner",
                    "source_url": full_url,
                    "raw_category_from_katalog": katalog["raw_category"],
                    "keyword": katalog["keyword"],
                    "city_from_katalog": katalog["city"]
                })

        except Exception as e:
            print(f"Failed katalog: {url} | {e}")

        time.sleep(REQUEST_DELAY)

    df_links = pd.DataFrame(collected)

    if df_links.empty:
        return df_links

    df_links = df_links.drop_duplicates(subset=["source_url"]).reset_index(drop=True)
    df_links = df_links.head(MAX_LINKS)

    return df_links


df_links = collect_menu_links()

print(df_links.shape)
display(df_links.head(20))

Skip https://menukuliner.net/katalog/kopi | status 404


Skip https://menukuliner.net/katalog/boba | status 404


Skip https://menukuliner.net/katalog/matcha | status 404


Skip https://menukuliner.net/katalog/milk-tea | status 404


Skip https://menukuliner.net/katalog/thai-tea | status 404


Skip https://menukuliner.net/katalog/es-kopi-susu | status 404
(75, 5)


,source_platform,source_url,raw_category_from_katalog,keyword,city_from_katalog
0,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee / boba,coffee boba,unknown
1,MenuKuliner,https://menukuliner.net/menu/40917/pot-o-koffi...,coffee / boba,coffee boba,unknown
2,MenuKuliner,https://menukuliner.net/menu/74731/bebe-bubble...,coffee / boba,coffee boba,unknown
3,MenuKuliner,https://menukuliner.net/menu/75034/bens-coffee...,coffee / boba,coffee boba,unknown
4,MenuKuliner,https://menukuliner.net/menu/80044/cochoc-band...,coffee / boba,coffee boba,unknown
5,MenuKuliner,https://menukuliner.net/menu/104601/maxxi-coff...,coffee / boba,coffee boba,unknown
6,MenuKuliner,https://menukuliner.net/menu/130481/teguk-cari...,coffee / boba,coffee boba,unknown
7,MenuKuliner,https://menukuliner.net/menu/130491/teguk-permata,coffee / boba,coffee boba,unknown
8,MenuKuliner,https://menukuliner.net/menu/183659/imahkopi-k...,coffee / boba,coffee boba,unknown
9,MenuKuliner,https://menukuliner.net/menu/264144/boba-time-...,coffee / boba,coffee boba,unknown


In [33]:
def get_restaurant_name(soup):
    h1 = soup.find("h1")

    if h1:
        title = clean_text(h1.get_text())
    else:
        title_tag = soup.find("title")
        title = clean_text(title_tag.get_text()) if title_tag else "Unknown Restaurant"

    title = title.replace("Daftar Harga Menu Delivery", "")
    title = title.replace("Daftar Harga Menu", "")
    title = title.replace("Terbaru", "")
    title = re.sub(r"\s+", " ", title)
    title = title.strip(" ,-")

    if title == "":
        return "Unknown Restaurant"

    return title


def extract_menu_from_table(soup):
    items = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = [clean_text(td.get_text()) for td in tr.find_all(["td", "th"])]

            if len(cells) < 2:
                continue

            name_candidate = cells[0]
            price_candidate = cells[-1]

            if not is_price(price_candidate):
                continue

            if not is_valid_menu_name(name_candidate):
                continue

            items.append({
                "section": None,
                "menu_name": name_candidate,
                "price": clean_price(price_candidate),
                "extract_method": "html_table"
            })

    return items


def extract_menu_from_text(soup):
    text = soup.get_text("\n")
    lines = [clean_text(line) for line in text.splitlines()]
    lines = [line for line in lines if line]

    items = []
    current_section = None

    for idx, line in enumerate(lines):
        if line.lower().startswith("harga menu"):
            current_section = clean_text(line.replace("Harga Menu", ""))
            continue

        if not is_price(line):
            continue

        price = clean_price(line)

        if pd.isna(price):
            continue

        candidates = []

        if idx - 1 >= 0:
            candidates.append(lines[idx - 1])

        if idx - 2 >= 0:
            candidates.append(lines[idx - 2])

        menu_name = None

        for candidate in candidates:
            if is_valid_menu_name(candidate):
                menu_name = candidate
                break

        if menu_name is None:
            continue

        items.append({
            "section": current_section,
            "menu_name": menu_name,
            "price": price,
            "extract_method": "text_pattern"
        })

    return items


def deduplicate_menu_items(items):
    unique = []
    seen = set()

    for item in items:
        if pd.isna(item["price"]):
            continue

        key = (
            str(item.get("section")).lower(),
            item["menu_name"].lower(),
            int(item["price"])
        )

        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique


def scrape_menu_page(target):
    rows = []
    url = target["source_url"]

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        restaurant_name = get_restaurant_name(soup)

        menu_items = []
        menu_items.extend(extract_menu_from_table(soup))
        menu_items.extend(extract_menu_from_text(soup))
        menu_items = deduplicate_menu_items(menu_items)

        city = infer_city_from_text(restaurant_name + " " + url)

        if city == "Unknown" and target["city_from_katalog"] != "unknown":
            city = str(target["city_from_katalog"]).title()

        for item in menu_items:
            raw_category = infer_raw_category(
                str(target["raw_category_from_katalog"]) + " " +
                str(target["keyword"]) + " " +
                str(item["section"]) + " " +
                str(item["menu_name"])
            )

            rows.append({
                "restaurant_name": restaurant_name,
                "city": city,
                "raw_category": raw_category,
                "section": item["section"],
                "menu_name": item["menu_name"],
                "price": item["price"],
                "source_platform": target["source_platform"],
                "source_url": url,
                "search_keyword": target["keyword"],
                "extract_method": item["extract_method"],
                "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })

    except Exception as e:
        print(f"Failed: {url} | {e}")

    return rows

In [34]:
all_rows = []

for _, target in tqdm(df_links.iterrows(), total=len(df_links), desc="Scraping menu pages"):
    rows = scrape_menu_page(target)
    all_rows.extend(rows)

    if all_rows:
        pd.DataFrame(all_rows).to_csv("checkpoint_cafe_drinks_scrape.csv", index=False)

    time.sleep(REQUEST_DELAY)

df = pd.DataFrame(all_rows)

if df.empty:
    print("Tidak ada data berhasil discrape.")
else:
    df = df.drop_duplicates(
        subset=["restaurant_name", "menu_name", "price", "source_url"],
        keep="first"
    ).reset_index(drop=True)

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = df[
        (df["price"].isna()) |
        ((df["price"] >= 3000) & (df["price"] <= 500000))
    ].copy()

    df = df.reset_index(drop=True)

    df.to_csv(OUTPUT_FILE, index=False)

    print("=" * 80)
    print("SCRAPING CAFE DRINKS SELESAI")
    print("=" * 80)
    print(f"Output file        : {OUTPUT_FILE}")
    print(f"Total rows         : {len(df)}")
    print(f"Unique restaurants : {df['restaurant_name'].nunique()}")
    print(f"Unique cities      : {df['city'].nunique()}")
    print(f"Source platform    : {df['source_platform'].unique().tolist()}")
    print(f"Min price          : {df['price'].min()}")
    print(f"Median price       : {df['price'].median()}")
    print(f"Max price          : {df['price'].max()}")

    display(df.head(30))

Scraping menu pages: 100%|██████████| 75/75 [02:37<00:00,  2.10s/it]


SCRAPING CAFE DRINKS SELESAI
Output file        : cafe_drinks_scrape.csv
Total rows         : 5330
Unique restaurants : 75
Unique cities      : 10
Source platform    : ['MenuKuliner']
Min price          : 3000
Median price       : 19000.0
Max price          : 330000


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
0,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Berdua Bahagia Berbagi Kebahagiaan Dengan Yang...,46000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
1,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Kompak Bertiga Bertiga Saling Menguatkan. Untu...,63000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
2,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Boba Coffee Milk Kopi Susu Dengan Boba Kenyal ...,14000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
3,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Coffee Milk Kopi Espresso Berkolaborasi Dengan...,14000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
4,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Coco Milk Coffee Andai Kamu Tahu Segarnya Kopi...,15000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
5,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Boba Choco Milk Racikan DarkCoklat Nikmat Dise...,14000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
6,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Choco Milk Racikan Dark Coklat Yang Kental Dan...,14000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
7,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Red Velvet Paduan Racikan Redvelvet Dengan Top...,15000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
8,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Milky Honey Freshmilk Dan Madu Murni Berpadu. ...,14000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58
9,"Kopi Semata, Anggrek, Yogyakarta 2026",Yogyakarta,boba,None,Milky Way Hanya Es Dan Susu Saja. Sangat Menye...,11000,MenuKuliner,https://menukuliner.net/menu/1076542/kopi-sema...,coffee boba,html_table,2026-04-28 02:55:58


In [35]:
print("Shape:", df.shape)

display(df["raw_category"].value_counts())
display(df["restaurant_name"].value_counts().head(20))
display(df.sample(min(20, len(df)), random_state=42))

Shape: (5330, 11)


,count
raw_category,
boba,5330


,count
restaurant_name,
"Ali Kopi Cafe & Roastery, Kalideres, Jakarta 2026",281
"Pot O Koffie, Grand City, Balikpapan 2026",232
"Baby K Food and Drink, Brigjend Katamso, Solo 2026",206
"Nyopee Medan, HM Yamin, Medan 2026",203
"Jellypotter Cibinong 2, Cibinong, Jakarta 2026",163
"Semoxjuice, Kahuripan, Malang 2026",160
"Lampion Cafe by Pasta Kangen, Teluknaga, Jakarta 2026",155
"Jelly Potter, Puri Nirwana 2, Jakarta 2026",147
"Maxxi Coffeebar, Sukaraja 2, Bandung 2026",140


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
1323,"Teguk, Condet, Jakarta 2026",Jakarta,boba,Boba Series,Gratis,15000,MenuKuliner,https://menukuliner.net/menu/562343/teguk-condet,coffee boba,text_pattern,2026-04-28 02:56:39
1839,"Jelly Potter, Puri Nirwana 2, Jakarta 2026",Jakarta,boba,None,Matcha Spesial Gula Aren Boba Dengan Topping Boba,13500,MenuKuliner,https://menukuliner.net/menu/348537/jelly-pott...,coffee boba,html_table,2026-04-28 02:56:59
798,"CO.CHOC, Bintaro, Jakarta 2026",Jakarta,boba,None,Chocolate Army Perpaduan Dark Chocolate yang d...,19500,MenuKuliner,https://menukuliner.net/menu/285680/cochoc-bin...,coffee boba,html_table,2026-04-28 02:56:21
3856,"Baby K Food and Drink, Brigjend Katamso, Solo ...",Solo,boba,Siap Masak,Batagor Kuah Siap Masak Dengan Bumbu Pedas Yan...,15000,MenuKuliner,https://menukuliner.net/menu/897260/baby-k-foo...,kopi boba,text_pattern,2026-04-28 02:57:52
4553,"Jelly Potter Muncang, Lagoa,koja.warteg Sualai...",Jakarta,boba,None,Swis Ovaltine Boba,15000,MenuKuliner,https://menukuliner.net/menu/348468/jelly-pott...,kopi boba,html_table,2026-04-28 02:58:11
856,"Haus!, Cipete, Jakarta 2026",Jakarta,boba,None,Lemon Tea Size: Small GratisLarge + Rp 5.000,7000,MenuKuliner,https://menukuliner.net/menu/337316/haus-cipete,coffee boba,html_table,2026-04-28 02:56:23
2333,"Kopi Kanto, Kelapa Gading, Jakarta 2026",Jakarta,boba,Kanto Sweet Snack,Puff Yang Lembut Dipadukan Dengan Skippy Chunk...,19000,MenuKuliner,https://menukuliner.net/menu/381666/kopi-kanto...,kopi boba,text_pattern,2026-04-28 02:57:15
2499,"Momimi, Damai, Balikpapan 2026",Balikpapan,boba,None,Tiramisu Crumble Ice Level: Less GratisNormal ...,21500,MenuKuliner,https://menukuliner.net/menu/40076/momimi-damai,kopi boba,html_table,2026-04-28 02:57:23
5011,"Lokale, Kuta Utara, Bali 2026",Unknown,boba,None,Kopi Matcha Kopi Susu + MatchaSize: Reguler Gr...,15000,MenuKuliner,https://menukuliner.net/menu/16453/lokale-kuta...,boba kopi,html_table,2026-04-28 02:58:26
4883,"Cafe Margaux Deli, Jakarta 2026",Jakarta,boba,Signature Non-Coffee,Gratis,44000,MenuKuliner,https://menukuliner.net/menu/275383/cafe-marga...,kopi boba,text_pattern,2026-04-28 02:58:20


In [37]:
from google.colab import files

files.download("cafe_drinks_scrape.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>